In [1]:

from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
import json
import sys
import time

import pandas as pd

# Discover the existing APEX project root.
current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "AGENTS.md").is_file()
        and (path / "README.md").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "APEX root not found. Open this notebook inside the project."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "jolpica" / "results"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

JOLPICA_BASE = "https://api.jolpi.ca/ergast/f1"
HEADERS = {"User-Agent": "APEX-F1-Research/1.0"}

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("Raw response directory:", RAW_DIR)
print("Interim dataset directory:", INTERIM_DIR)


def fetch_jolpica(endpoint, params=None, timeout=20):
    """Fetch one Jolpica JSON response and return status, data, latency."""
    params = params or {}
    query = urlencode(params)
    url = f"{JOLPICA_BASE}/{endpoint}"
    if query:
        url = f"{url}?{query}"

    request = Request(url, headers=HEADERS)
    started = time.perf_counter()

    try:
        with urlopen(request, timeout=timeout) as response:
            payload = json.loads(response.read().decode("utf-8"))
            status = response.status
    except HTTPError as exc:
        status = exc.code
        try:
            payload = exc.read().decode("utf-8", errors="replace")[:1000]
        except Exception:
            payload = str(exc)
    except (URLError, TimeoutError, json.JSONDecodeError) as exc:
        status = 0
        payload = f"{type(exc).__name__}: {exc}"

    latency_ms = (time.perf_counter() - started) * 1000
    return status, payload, latency_ms


Project root: /Users/dhruv/dhruv files/apex-f1
Python: /opt/anaconda3/bin/python
Pandas: 2.2.2
Raw response directory: /Users/dhruv/dhruv files/apex-f1/data/raw/jolpica/results
Interim dataset directory: /Users/dhruv/dhruv files/apex-f1/data/interim


In [2]:

# Pilot dataset: 2024 Grand Prix race results only.
# Sprint results are intentionally excluded from this first dataset.

SEASON = 2024
PAGE_SIZE = 100
offset = 0

race_metadata = {}
results_by_round = {}
page_audit = []

while True:
    status, payload, latency_ms = fetch_jolpica(
        f"{SEASON}/results.json",
        params={"limit": PAGE_SIZE, "offset": offset},
    )

    if status != 200 or not isinstance(payload, dict):
        raise RuntimeError(
            f"Jolpica request failed at offset={offset}: "
            f"HTTP {status} | {payload}"
        )

    mrdata = payload.get("MRData", {})
    race_table = mrdata.get("RaceTable", {})
    races = race_table.get("Races", [])

    api_total = int(mrdata.get("total", 0))
    api_limit = int(mrdata.get("limit", PAGE_SIZE))

    if api_limit <= 0:
        raise RuntimeError(f"Invalid API page limit: {api_limit}")

    page_rows = 0

    # Merge results by race round and driver, not simply by page.
    for race in races:
        round_number = int(race["round"])
        race_metadata[round_number] = {
            key: value
            for key, value in race.items()
            if key != "Results"
        }

        round_results = results_by_round.setdefault(round_number, {})

        for result in race.get("Results", []):
            driver = result.get("Driver", {})
            driver_id = driver.get("driverId")

            if driver_id is None:
                raise ValueError(
                    f"Missing driverId in round {round_number}"
                )

            round_results[driver_id] = result
            page_rows += 1

    # Preserve raw API responses for reproducibility.
    raw_file = RAW_DIR / f"{SEASON}_offset_{offset:04d}.json"
    raw_file.write_text(
        json.dumps(payload, indent=2),
        encoding="utf-8",
    )

    page_audit.append({
        "season": SEASON,
        "offset": offset,
        "api_limit": api_limit,
        "api_total": api_total,
        "races_in_response": len(races),
        "result_rows_in_response": page_rows,
        "http_status": status,
        "latency_ms": round(latency_ms, 1),
    })

    print(
        f"offset={offset:>4} | "
        f"HTTP {status} | "
        f"races={len(races):>2} | "
        f"rows={page_rows:>3} | "
        f"API total={api_total} | "
        f"{latency_ms:.0f} ms"
    )

    if offset + api_limit >= api_total:
        break

    offset += api_limit
    time.sleep(0.4)

# Summarize actual coverage.
coverage = pd.DataFrame([
    {
        "season": SEASON,
        "round": round_number,
        "race_name": race_metadata[round_number].get("raceName"),
        "drivers_with_results": len(results_by_round[round_number]),
        "scheduled_date": race_metadata[round_number].get("date"),
    }
    for round_number in sorted(race_metadata)
])

page_audit_df = pd.DataFrame(page_audit)

display(coverage)
display(page_audit_df)

print("Unique Grand Prix rounds:", coverage["round"].nunique())
print(
    "Unique driver-race records:",
    sum(len(rows) for rows in results_by_round.values()),
)
print("Missing round numbers in 2024:",
      sorted(set(range(1, 25)) - set(coverage["round"])))
print("Raw API pages saved:", len(page_audit))


offset=   0 | HTTP 200 | races= 6 | rows=100 | API total=479 | 1280 ms
offset= 100 | HTTP 200 | races= 6 | rows=100 | API total=479 | 2288 ms
offset= 200 | HTTP 200 | races= 6 | rows=100 | API total=479 | 1640 ms
offset= 300 | HTTP 200 | races= 6 | rows=100 | API total=479 | 1375 ms
offset= 400 | HTTP 200 | races= 4 | rows= 79 | API total=479 | 1375 ms


,season,round,race_name,drivers_with_results,scheduled_date
0,2024,1,Bahrain Grand Prix,20,2024-03-02
1,2024,2,Saudi Arabian Grand Prix,20,2024-03-09
2,2024,3,Australian Grand Prix,19,2024-03-24
3,2024,4,Japanese Grand Prix,20,2024-04-07
4,2024,5,Chinese Grand Prix,20,2024-04-21
5,2024,6,Miami Grand Prix,20,2024-05-05
6,2024,7,Emilia Romagna Grand Prix,20,2024-05-19
7,2024,8,Monaco Grand Prix,20,2024-05-26
8,2024,9,Canadian Grand Prix,20,2024-06-09
9,2024,10,Spanish Grand Prix,20,2024-06-23


,season,offset,api_limit,api_total,races_in_response,result_rows_in_response,http_status,latency_ms
0,2024,0,100,479,6,100,200,1279.5
1,2024,100,100,479,6,100,200,2287.9
2,2024,200,100,479,6,100,200,1639.7
3,2024,300,100,479,6,100,200,1375.2
4,2024,400,100,479,4,79,200,1375.4


Unique Grand Prix rounds: 24
Unique driver-race records: 479
Missing round numbers in 2024: []
Raw API pages saved: 5


In [3]:

import json
import pandas as pd

# Normalize the raw Jolpica response records into one driver–race table.
rows = []

for round_number in sorted(results_by_round):
    race = race_metadata[round_number]
    circuit = race.get("Circuit", {})
    location = circuit.get("Location", {})

    for driver_id, result in results_by_round[round_number].items():
        driver = result.get("Driver", {})
        constructor = result.get("Constructor", {})
        time_info = result.get("Time", {}) or {}
        fastest_lap = result.get("FastestLap", {}) or {}
        lap_time = fastest_lap.get("Time", {}) or {}
        avg_speed = fastest_lap.get("AverageSpeed", {}) or {}

        rows.append({
            "season": int(race["season"]),
            "round": int(race["round"]),
            "race_name": race.get("raceName"),
            "race_date": race.get("date"),
            "circuit_id": circuit.get("circuitId"),
            "circuit_name": circuit.get("circuitName"),
            "country": location.get("country"),
            "driver_id": driver_id,
            "driver_code": driver.get("code"),
            "driver_name": (
                f"{driver.get('givenName', '')} "
                f"{driver.get('familyName', '')}"
            ).strip(),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "grid_position": result.get("grid"),
            "finish_position": result.get("position"),
            "position_text": result.get("positionText"),
            "points": result.get("points"),
            "laps_completed": result.get("laps"),
            "status": result.get("status"),
            "official_time": time_info.get("time"),
            "time_millis": time_info.get("millis"),
            "fastest_lap_number": fastest_lap.get("lap"),
            "fastest_lap_time": lap_time.get("time"),
            "fastest_lap_rank": fastest_lap.get("rank"),
            "fastest_lap_avg_speed": avg_speed.get("speed"),
        })

df_2024 = pd.DataFrame(rows)

# Parse types explicitly; unparseable finishing-position text becomes missing.
numeric_columns = [
    "season", "round", "grid_position", "finish_position",
    "points", "laps_completed", "time_millis",
    "fastest_lap_number", "fastest_lap_rank", "fastest_lap_avg_speed"
]

for column in numeric_columns:
    df_2024[column] = pd.to_numeric(
        df_2024[column], errors="coerce"
    )

df_2024["race_date"] = pd.to_datetime(
    df_2024["race_date"], errors="coerce"
)

# Validate the fundamental driver–race key.
key_columns = ["season", "round", "driver_id"]
duplicate_mask = df_2024.duplicated(key_columns, keep=False)

print("Dataset shape:", df_2024.shape)
print("Unique rounds:", df_2024["round"].nunique())
print("Unique drivers:", df_2024["driver_id"].nunique())
print("Duplicate driver–race keys:", int(duplicate_mask.sum()))
print("Missing driver IDs:", int(df_2024["driver_id"].isna().sum()))

assert len(df_2024) == 479, "Row count differs from the audited result records."
assert df_2024["round"].nunique() == 24, "Expected 24 race rounds."
assert duplicate_mask.sum() == 0, "Duplicate driver–race keys detected."
assert df_2024["driver_id"].notna().all(), "Missing driver IDs detected."

# Report data quality without silently deleting records.
quality_columns = [
    "driver_name", "constructor_name", "grid_position",
    "finish_position", "points", "laps_completed", "status"
]

quality_report = pd.DataFrame({
    "missing_count": df_2024[quality_columns].isna().sum(),
    "missing_percent": (
        df_2024[quality_columns].isna().mean().mul(100).round(2)
    ),
})

display(quality_report)
display(df_2024.head(10))

# Save the initial research dataset and its provenance manifest.
output_file = INTERIM_DIR / "driver_race_results_2024_v1.csv"
df_2024.to_csv(output_file, index=False)

manifest = {
    "dataset_name": "APEX 2024 driver-race results",
    "dataset_version": "v1",
    "source": "Jolpica F1 API, Ergast-compatible",
    "season": 2024,
    "endpoint": "2024/results.json",
    "raw_pages": len(page_audit),
    "row_count": len(df_2024),
    "unique_race_rounds": int(df_2024["round"].nunique()),
    "unique_drivers": int(df_2024["driver_id"].nunique()),
    "primary_key": ["season", "round", "driver_id"],
    "duplicate_primary_keys": int(duplicate_mask.sum()),
    "contains_model_predictions": False,
    "notes": [
        "Sprint results are excluded.",
        "Classification status is preserved separately.",
        "No DNF labels have been derived yet.",
        "Missing values are reported, not silently removed."
    ],
}

manifest_file = INTERIM_DIR / "driver_race_results_2024_v1.manifest.json"
manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\nSaved dataset:", output_file)
print("Saved manifest:", manifest_file)


Dataset shape: (479, 24)
Unique rounds: 24
Unique drivers: 24
Duplicate driver–race keys: 0
Missing driver IDs: 0


,missing_count,missing_percent
driver_name,0,0.0
constructor_name,0,0.0
grid_position,0,0.0
finish_position,0,0.0
points,0,0.0
laps_completed,0,0.0
status,0,0.0


,season,round,race_name,race_date,circuit_id,circuit_name,country,driver_id,driver_code,driver_name,...,position_text,points,laps_completed,status,official_time,time_millis,fastest_lap_number,fastest_lap_time,fastest_lap_rank,fastest_lap_avg_speed
0,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,max_verstappen,VER,Max Verstappen,...,1,26,57,Finished,1:31:44.742,5504742.0,39.0,1:32.608,1.0,210.383
1,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,perez,PER,Sergio Pérez,...,2,18,57,Finished,+22.457,5527199.0,40.0,1:34.364,4.0,206.468
2,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,sainz,SAI,Carlos Sainz,...,3,15,57,Finished,+25.110,5529852.0,44.0,1:34.507,6.0,206.156
3,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,leclerc,LEC,Charles Leclerc,...,4,12,57,Finished,+39.669,5544411.0,36.0,1:34.090,2.0,207.069
4,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,russell,RUS,George Russell,...,5,10,57,Finished,+46.788,5551530.0,40.0,1:35.065,12.0,204.946
5,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,norris,NOR,Lando Norris,...,6,8,57,Finished,+48.458,5553200.0,35.0,1:34.476,5.0,NaN
6,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,hamilton,HAM,Lewis Hamilton,...,7,6,57,Finished,+50.324,5555066.0,39.0,1:34.722,7.0,205.688
7,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,piastri,PIA,Oscar Piastri,...,8,4,57,Finished,+56.082,5560824.0,39.0,1:34.983,11.0,NaN
8,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,alonso,ALO,Fernando Alonso,...,9,2,57,Finished,+1:14.887,5579629.0,48.0,1:34.199,3.0,206.830
9,2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,stroll,STR,Lance Stroll,...,10,1,57,Finished,+1:33.216,5597958.0,30.0,1:35.632,16.0,203.730



Saved dataset: /Users/dhruv/dhruv files/apex-f1/data/interim/driver_race_results_2024_v1.csv
Saved manifest: /Users/dhruv/dhruv files/apex-f1/data/interim/driver_race_results_2024_v1.manifest.json


In [4]:

# APEX — 2024 Pilot Dataset: Deeper Quality Audit

import pandas as pd

race_keys = ["season", "round"]
driver_race_key = ["season", "round", "driver_id"]

# 1. Verify the primary key.
duplicate_rows = df_2024.duplicated(driver_race_key).sum()

# 2. Check that every race has exactly one recorded winner.
df_2024["is_winner"] = df_2024["finish_position"].eq(1)

winner_summary = (
    df_2024.groupby(race_keys)
    .agg(
        result_records=("driver_id", "size"),
        unique_drivers=("driver_id", "nunique"),
        winner_count=("is_winner", "sum"),
    )
    .reset_index()
)

winner_rows = df_2024.loc[
    df_2024["is_winner"],
    ["season", "round", "race_name", "driver_name", "constructor_name"]
]

winner_summary = winner_summary.merge(
    winner_rows.rename(columns={
        "driver_name": "winner",
        "constructor_name": "winning_constructor",
    }),
    on=race_keys,
    how="left",
    validate="one_to_one",
)

display(winner_summary)

# 3. Inspect missingness across ALL columns, not just selected fields.
missingness = pd.DataFrame({
    "missing_count": df_2024.isna().sum(),
    "missing_percent": (df_2024.isna().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)

display(missingness)

# 4. Inspect official result-status semantics.
print("Most frequent official statuses:")
display(
    df_2024["status"]
    .value_counts(dropna=False)
    .rename_axis("official_status")
    .reset_index(name="records")
    .head(20)
)

print("\nPosition-text distribution:")
display(
    df_2024["position_text"]
    .value_counts(dropna=False)
    .rename_axis("position_text")
    .reset_index(name="records")
)

# 5. Check plausible numeric values.
numeric_checks = {
    "negative points": int((df_2024["points"] < 0).sum()),
    "negative completed laps": int((df_2024["laps_completed"] < 0).sum()),
    "negative grid positions": int((df_2024["grid_position"] < 0).sum()),
    "missing numeric finish positions": int(
        df_2024["finish_position"].isna().sum()
    ),
}

print("Numeric checks:")
for check, count in numeric_checks.items():
    print(f"  {check}: {count}")

# 6. Summarize the audit without silently dropping records.
print("\nAudit summary")
print("Rows:", len(df_2024))
print("Rounds:", df_2024["round"].nunique())
print("Duplicate driver-race rows:", int(duplicate_rows))
print("Races with exactly one winner:",
      int(winner_summary["winner_count"].eq(1).sum()))
print("Races with zero or multiple winners:",
      int(winner_summary["winner_count"].ne(1).sum()))

assert duplicate_rows == 0
assert winner_summary["winner_count"].eq(1).all()
assert set(df_2024["round"].unique()) == set(range(1, 25))


,season,round,result_records,unique_drivers,winner_count,race_name,winner,winning_constructor
0,2024,1,20,20,1,Bahrain Grand Prix,Max Verstappen,Red Bull
1,2024,2,20,20,1,Saudi Arabian Grand Prix,Max Verstappen,Red Bull
2,2024,3,19,19,1,Australian Grand Prix,Carlos Sainz,Ferrari
3,2024,4,20,20,1,Japanese Grand Prix,Max Verstappen,Red Bull
4,2024,5,20,20,1,Chinese Grand Prix,Max Verstappen,Red Bull
5,2024,6,20,20,1,Miami Grand Prix,Lando Norris,McLaren
6,2024,7,20,20,1,Emilia Romagna Grand Prix,Max Verstappen,Red Bull
7,2024,8,20,20,1,Monaco Grand Prix,Charles Leclerc,Ferrari
8,2024,9,20,20,1,Canadian Grand Prix,Max Verstappen,Red Bull
9,2024,10,20,20,1,Spanish Grand Prix,Max Verstappen,Red Bull


,missing_count,missing_percent
time_millis,47,9.81
official_time,47,9.81
fastest_lap_avg_speed,45,9.39
fastest_lap_rank,17,3.55
fastest_lap_time,17,3.55
fastest_lap_number,17,3.55
season,0,0.00
finish_position,0,0.00
status,0,0.00
laps_completed,0,0.00


Most frequent official statuses:


,official_status,records
0,Finished,287
1,Lapped,138
2,Retired,49
3,Did not start,3
4,Disqualified,2



Position-text distribution:


,position_text,records
0,R,43
1,1,24
2,10,24
3,15,24
4,14,24
5,13,24
6,2,24
7,11,24
8,12,24
9,9,24


Numeric checks:
  negative points: 0
  negative completed laps: 0
  negative grid positions: 0
  missing numeric finish positions: 0

Audit summary
Rows: 479
Rounds: 24
Duplicate driver-race rows: 0
Races with exactly one winner: 24
Races with zero or multiple winners: 0


In [5]:

import json
import time
import pandas as pd

START_SEASON = 2010
END_SEASON = 2026
PAGE_SIZE = 100

def collect_season(season):
    """Fetch and deduplicate one season's Grand Prix race results."""
    season_dir = RAW_DIR / str(season)
    season_dir.mkdir(parents=True, exist_ok=True)

    records_by_key = {}
    race_metadata = {}
    page_log = []
    offset = 0
    expected_total = None

    while True:
        status, payload, latency_ms = fetch_jolpica(
            f"{season}/results.json",
            params={"limit": PAGE_SIZE, "offset": offset},
        )

        if status != 200 or not isinstance(payload, dict):
            raise RuntimeError(
                f"{season}: request failed at offset={offset}; "
                f"HTTP {status}: {str(payload)[:300]}"
            )

        mrdata = payload.get("MRData", {})
        races = mrdata.get("RaceTable", {}).get("Races", [])

        api_total = int(mrdata.get("total", 0))
        api_limit = int(mrdata.get("limit", PAGE_SIZE))
        expected_total = api_total

        if api_limit <= 0:
            raise RuntimeError(
                f"{season}: invalid page limit {api_limit}"
            )

        # Save the unmodified response for reproducibility.
        raw_file = season_dir / f"results_offset_{offset:04d}.json"
        raw_file.write_text(
            json.dumps(payload, indent=2),
            encoding="utf-8",
        )

        rows_in_page = 0

        for race in races:
            round_number = int(race["round"])

            race_metadata[round_number] = {
                key: value
                for key, value in race.items()
                if key != "Results"
            }

            for result in race.get("Results", []):
                driver = result.get("Driver", {})
                driver_id = driver.get("driverId")

                if not driver_id:
                    raise ValueError(
                        f"{season} round {round_number}: missing driverId"
                    )

                key = (season, round_number, driver_id)
                records_by_key[key] = {
                    "race": race_metadata[round_number],
                    "result": result,
                }
                rows_in_page += 1

        page_log.append({
            "season": season,
            "offset": offset,
            "http_status": status,
            "api_total_records": api_total,
            "races_in_response": len(races),
            "rows_in_response": rows_in_page,
            "latency_ms": round(latency_ms, 1),
        })

        if season == START_SEASON or offset == 0:
            print(
                f"{season}: page offset={offset}, "
                f"rows={rows_in_page}, API total={api_total}"
            )

        # Stop safely for an empty response or after the final page.
        if api_total == 0 or not races:
            break

        if offset + api_limit >= api_total:
            break

        offset += api_limit
        time.sleep(0.4)

    unique_records = []
    for (record_season, round_number, driver_id), item in (
        records_by_key.items()
    ):
        race = item["race"]
        result = item["result"]
        driver = result.get("Driver", {})
        constructor = result.get("Constructor", {})
        circuit = race.get("Circuit", {})
        location = circuit.get("Location", {})
        time_info = result.get("Time", {}) or {}
        fastest_lap = result.get("FastestLap", {}) or {}
        lap_time = fastest_lap.get("Time", {}) or {}
        avg_speed = fastest_lap.get("AverageSpeed", {}) or {}

        unique_records.append({
            "season": record_season,
            "round": round_number,
            "race_name": race.get("raceName"),
            "race_date": race.get("date"),
            "circuit_id": circuit.get("circuitId"),
            "circuit_name": circuit.get("circuitName"),
            "country": location.get("country"),
            "driver_id": driver_id,
            "driver_code": driver.get("code"),
            "driver_name": (
                f"{driver.get('givenName', '')} "
                f"{driver.get('familyName', '')}"
            ).strip(),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "grid_position": result.get("grid"),
            "finish_position": result.get("position"),
            "position_text": result.get("positionText"),
            "points": result.get("points"),
            "laps_completed": result.get("laps"),
            "status": result.get("status"),
            "official_time": time_info.get("time"),
            "time_millis": time_info.get("millis"),
            "fastest_lap_number": fastest_lap.get("lap"),
            "fastest_lap_time": lap_time.get("time"),
            "fastest_lap_rank": fastest_lap.get("rank"),
            "fastest_lap_avg_speed": avg_speed.get("speed"),
        })

    result_df = pd.DataFrame(unique_records)

    return result_df, page_log, expected_total


season_frames = []
all_page_logs = []
season_audit = []

for season in range(START_SEASON, END_SEASON + 1):
    time.sleep(0.4)

    season_df, pages, expected_total = collect_season(season)
    season_frames.append(season_df)
    all_page_logs.extend(pages)

    unique_records = len(season_df)
    season_audit.append({
        "season": season,
        "api_expected_records": expected_total,
        "unique_records": unique_records,
        "rounds": (
            int(season_df["round"].nunique())
            if not season_df.empty else 0
        ),
        "record_count_matches_api": (
            unique_records == expected_total
        ),
    })

    print(
        f"Completed {season}: {unique_records} unique records "
        f"across {season_audit[-1]['rounds']} rounds"
    )

all_results_df = pd.concat(season_frames, ignore_index=True)
page_audit_df = pd.DataFrame(all_page_logs)
season_audit_df = pd.DataFrame(season_audit)

# Normalize data types.
numeric_columns = [
    "season", "round", "grid_position", "finish_position",
    "points", "laps_completed", "time_millis",
    "fastest_lap_number", "fastest_lap_rank",
    "fastest_lap_avg_speed",
]
for column in numeric_columns:
    all_results_df[column] = pd.to_numeric(
        all_results_df[column], errors="coerce"
    )

all_results_df["race_date"] = pd.to_datetime(
    all_results_df["race_date"], errors="coerce"
)

# Check the intended primary key across the full dataset.
primary_key = ["season", "round", "driver_id"]
duplicate_count = int(
    all_results_df.duplicated(primary_key).sum()
)

print("\nSeason coverage audit:")
display(season_audit_df)

print("\nFull dataset overview:")
print("Rows:", len(all_results_df))
print("Seasons:", all_results_df["season"].nunique())
print("Rounds:", all_results_df[["season", "round"]].drop_duplicates().shape[0])
print("Unique drivers:", all_results_df["driver_id"].nunique())
print("Duplicate primary keys:", duplicate_count)

# Save data and audit artifacts.
dataset_path = (
    INTERIM_DIR / f"driver_race_results_{START_SEASON}_{END_SEASON}_v1.csv"
)
all_results_df.to_csv(dataset_path, index=False)

season_audit_path = (
    INTERIM_DIR / f"season_coverage_{START_SEASON}_{END_SEASON}_v1.csv"
)
season_audit_df.to_csv(season_audit_path, index=False)

page_audit_path = (
    INTERIM_DIR / f"api_page_audit_{START_SEASON}_{END_SEASON}_v1.csv"
)
page_audit_df.to_csv(page_audit_path, index=False)

print("\nSaved dataset:", dataset_path)
print("Saved season audit:", season_audit_path)
print("Saved API page audit:", page_audit_path)


2010: page offset=0, rows=100, API total=456
2010: page offset=100, rows=100, API total=456
2010: page offset=200, rows=100, API total=456
2010: page offset=300, rows=100, API total=456
2010: page offset=400, rows=56, API total=456
Completed 2010: 456 unique records across 19 rounds
2011: page offset=0, rows=100, API total=454
Completed 2011: 454 unique records across 19 rounds
2012: page offset=0, rows=100, API total=478
Completed 2012: 478 unique records across 20 rounds
2013: page offset=0, rows=100, API total=418
Completed 2013: 418 unique records across 19 rounds
2014: page offset=0, rows=100, API total=407
Completed 2014: 407 unique records across 19 rounds
2015: page offset=0, rows=100, API total=378
Completed 2015: 378 unique records across 19 rounds
2016: page offset=0, rows=100, API total=462
Completed 2016: 462 unique records across 21 rounds
2017: page offset=0, rows=100, API total=400
Completed 2017: 400 unique records across 20 rounds
2018: page offset=0, rows=100, API to

,season,api_expected_records,unique_records,rounds,record_count_matches_api
0,2010,456,456,19,True
1,2011,454,454,19,True
2,2012,478,478,20,True
3,2013,418,418,19,True
4,2014,407,407,19,True
5,2015,378,378,19,True
6,2016,462,462,21,True
7,2017,400,400,20,True
8,2018,420,420,21,True
9,2019,420,420,21,True



Full dataset overview:
Rows: 7285
Seasons: 17
Rounds: 346
Unique drivers: 84
Duplicate primary keys: 0

Saved dataset: /Users/dhruv/dhruv files/apex-f1/data/interim/driver_race_results_2010_2026_v1.csv
Saved season audit: /Users/dhruv/dhruv files/apex-f1/data/interim/season_coverage_2010_2026_v1.csv
Saved API page audit: /Users/dhruv/dhruv files/apex-f1/data/interim/api_page_audit_2010_2026_v1.csv


In [6]:

# APEX — Regulatory Era Metadata
# Metadata only: this does not yet alter model features or ratings.

import pandas as pd

def assign_regulation_era(season):
    if 2010 <= season <= 2013:
        return "2010_2013_v8_era"
    elif 2014 <= season <= 2021:
        return "2014_2021_hybrid_era"
    elif 2022 <= season <= 2025:
        return "2022_2025_ground_effect_era"
    elif season >= 2026:
        return "2026_new_technical_era"
    else:
        return "pre_2010_out_of_scope"

all_results_df["regulation_era"] = (
    all_results_df["season"].astype(int).map(assign_regulation_era)
)

# The FIA's April 2026 refinements were implemented from Miami,
# except for race-start changes that required further review.
miami_2026_date = pd.Timestamp("2026-05-03")

all_results_df["regulation_phase"] = "historical_phase_not_finely_coded"

mask_2026 = all_results_df["season"].eq(2026)

all_results_df.loc[
    mask_2026 & (all_results_df["race_date"] < miami_2026_date),
    "regulation_phase",
] = "2026_initial_regulatory_phase"

all_results_df.loc[
    mask_2026 & (all_results_df["race_date"] >= miami_2026_date),
    "regulation_phase",
] = "2026_post_miami_refinements"

print("Regulation eras:")
display(
    all_results_df.groupby(
        ["season", "regulation_era", "regulation_phase"],
        dropna=False,
    ).size().reset_index(name="driver_race_records")
)

# Every season should have an era classification.
assert all_results_df["regulation_era"].notna().all()


Regulation eras:


,season,regulation_era,regulation_phase,driver_race_records
0,2010,2010_2013_v8_era,historical_phase_not_finely_coded,456
1,2011,2010_2013_v8_era,historical_phase_not_finely_coded,454
2,2012,2010_2013_v8_era,historical_phase_not_finely_coded,478
3,2013,2010_2013_v8_era,historical_phase_not_finely_coded,418
4,2014,2014_2021_hybrid_era,historical_phase_not_finely_coded,407
5,2015,2014_2021_hybrid_era,historical_phase_not_finely_coded,378
6,2016,2014_2021_hybrid_era,historical_phase_not_finely_coded,462
7,2017,2014_2021_hybrid_era,historical_phase_not_finely_coded,400
8,2018,2014_2021_hybrid_era,historical_phase_not_finely_coded,420
9,2019,2014_2021_hybrid_era,historical_phase_not_finely_coded,420


In [7]:

import json
import pandas as pd

# APEX — Final validation for the regulation-aware dataset

required_columns = [
    "season",
    "round",
    "race_date",
    "driver_id",
    "finish_position",
    "status",
    "regulation_era",
    "regulation_phase",
]

missing_columns = [
    col for col in required_columns
    if col not in all_results_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

primary_key = ["season", "round", "driver_id"]
duplicate_count = int(all_results_df.duplicated(primary_key).sum())

# Verify every race has exactly one winner in the collected result data.
winner_check = (
    all_results_df.assign(
        _is_winner=all_results_df["finish_position"].eq(1)
    )
    .groupby(["season", "round"])["_is_winner"]
    .sum()
)

bad_winner_races = winner_check[winner_check != 1]

print("Total records:", len(all_results_df))
print("Season range:",
      int(all_results_df["season"].min()),
      "to",
      int(all_results_df["season"].max()))
print("Unique race weekends:",
      all_results_df[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate driver-race keys:", duplicate_count)
print("Races without exactly one recorded winner:",
      len(bad_winner_races))

print("\nRegulation-phase record counts:")
display(
    all_results_df.groupby(
        ["season", "regulation_era", "regulation_phase"],
        dropna=False,
    )
    .agg(
        records=("driver_id", "size"),
        rounds=("round", "nunique"),
    )
    .reset_index()
)

# Verify the 2026 split from the current dataset.
phase_2026 = (
    all_results_df.loc[all_results_df["season"] == 2026]
    .groupby("regulation_phase")
    .size()
)

print("\n2026 phase totals:")
display(phase_2026)

assert duplicate_count == 0, "Duplicate driver-race keys detected."
assert all_results_df["regulation_era"].notna().all()
assert all_results_df["regulation_phase"].notna().all()
assert bad_winner_races.empty, "Race winner validation failed."

# Save a new version; preserve the original pilot and raw API pages.
dataset_path = (
    INTERIM_DIR
    / "driver_race_results_2010_2026_v2_regulation_metadata.csv"
)
all_results_df.to_csv(dataset_path, index=False)

manifest = {
    "dataset_name": "APEX historical driver-race results",
    "dataset_version": "v2",
    "seasons": [
        int(all_results_df["season"].min()),
        int(all_results_df["season"].max()),
    ],
    "row_count": int(len(all_results_df)),
    "race_weekends": int(
        all_results_df[["season", "round"]].drop_duplicates().shape[0]
    ),
    "primary_key": primary_key,
    "duplicate_primary_keys": duplicate_count,
    "regulation_metadata": "Provisional, date-based grouping",
    "2026_result_records": int(
        all_results_df["season"].eq(2026).sum()
    ),
    "source": "Jolpica F1 API",
    "prediction_features_created": False,
    "limitations": [
        "Sprint results are excluded.",
        "Regulatory phase labels are provisional.",
        "Race outcomes must not be used as predictors for pre-race forecasts.",
    ],
}

manifest_path = dataset_path.with_suffix(".manifest.json")
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("\nSaved:", dataset_path)
print("Saved manifest:", manifest_path)


Total records: 7285
Season range: 2010 to 2026
Unique race weekends: 346
Duplicate driver-race keys: 0
Races without exactly one recorded winner: 0

Regulation-phase record counts:


,season,regulation_era,regulation_phase,records,rounds
0,2010,2010_2013_v8_era,historical_phase_not_finely_coded,456,19
1,2011,2010_2013_v8_era,historical_phase_not_finely_coded,454,19
2,2012,2010_2013_v8_era,historical_phase_not_finely_coded,478,20
3,2013,2010_2013_v8_era,historical_phase_not_finely_coded,418,19
4,2014,2014_2021_hybrid_era,historical_phase_not_finely_coded,407,19
5,2015,2014_2021_hybrid_era,historical_phase_not_finely_coded,378,19
6,2016,2014_2021_hybrid_era,historical_phase_not_finely_coded,462,21
7,2017,2014_2021_hybrid_era,historical_phase_not_finely_coded,400,20
8,2018,2014_2021_hybrid_era,historical_phase_not_finely_coded,420,21
9,2019,2014_2021_hybrid_era,historical_phase_not_finely_coded,420,21



2026 phase totals:


regulation_phase
2026_initial_regulatory_phase     66
2026_post_miami_refinements      308
dtype: int64


Saved: /Users/dhruv/dhruv files/apex-f1/data/interim/driver_race_results_2010_2026_v2_regulation_metadata.csv
Saved manifest: /Users/dhruv/dhruv files/apex-f1/data/interim/driver_race_results_2010_2026_v2_regulation_metadata.manifest.json


In [8]:

# APEX — Race Entrant and Grid Audit

import pandas as pd

print("Official result statuses:")
display(
    all_results_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nGrid-position distribution by status:")
grid_status_audit = pd.crosstab(
    all_results_df["status"],
    all_results_df["grid_position"].eq(0).rename("grid_position_is_zero"),
    margins=True,
)
display(grid_status_audit)

print("\nPotential non-starters:")
non_starters = all_results_df.loc[
    all_results_df["status"].astype(str).str.contains(
        r"did not start|withdrawn", case=False, regex=True
    ),
    [
        "season", "round", "race_name", "driver_name",
        "grid_position", "finish_position", "position_text", "status",
    ],
].sort_values(["season", "round"])

display(non_starters.head(30))
print("Potential non-starter rows:", len(non_starters))

print("\nGrid position zero examples:")
display(
    all_results_df.loc[
        all_results_df["grid_position"].eq(0),
        [
            "season", "round", "race_name", "driver_name",
            "grid_position", "finish_position", "position_text", "status",
        ],
    ].head(30)
)


Official result statuses:


,status,records
0,Finished,3822
1,+1 Lap,1342
2,Lapped,406
3,+2 Laps,314
4,Retired,279
...,...,...
73,+42 Laps,1
74,Pneumatics,1
75,+11 Laps,1
76,Injury,1



Grid-position distribution by status:


grid_position_is_zero,False,True,All
status,,,
+1 Lap,1324,18,1342
+10 Laps,2,0,2
+11 Laps,1,0,1
+14 Laps,1,0,1
+2 Laps,308,6,314
...,...,...,...
Water pump,1,0,1
Wheel,21,0,21
Wheel nut,8,0,8



Potential non-starters:


,season,round,race_name,driver_name,grid_position,finish_position,position_text,status
5792,2023,14,Italian Grand Prix,Yuki Tsunoda,11,20,W,Did not start
5852,2023,17,Qatar Grand Prix,Carlos Sainz,12,20,W,Did not start
5912,2023,20,São Paulo Grand Prix,Charles Leclerc,2,20,W,Did not start
6191,2024,12,British Grand Prix,Pierre Gasly,19,20,W,Did not start
6369,2024,21,São Paulo Grand Prix,Alexander Albon,7,18,W,Did not start
6370,2024,21,São Paulo Grand Prix,Lance Stroll,10,19,W,Did not start
6650,2025,11,Austrian Grand Prix,Carlos Sainz,19,20,W,Did not start
6670,2025,12,British Grand Prix,Franco Colapinto,20,20,W,Did not start
6750,2025,16,Italian Grand Prix,Nico Hülkenberg,12,20,W,Did not start
6931,2026,1,Australian Grand Prix,Oscar Piastri,5,21,W,Did not start


Potential non-starter rows: 16

Grid position zero examples:


,season,round,race_name,driver_name,grid_position,finish_position,position_text,status
597,2011,6,Monaco Grand Prix,Sergio Pérez,0,24,W,Injury
1099,2012,8,European Grand Prix,Timo Glock,0,24,W,Withdrew
2230,2015,1,Australian Grand Prix,Valtteri Bottas,0,18,W,Withdrew
2250,2015,2,Malaysian Grand Prix,Will Stevens,0,20,W,Fuel system
2320,2015,6,Monaco Grand Prix,Carlos Sainz,0,10,10,Finished
2717,2016,6,Monaco Grand Prix,Felipe Nasr,0,17,R,Retired
2718,2016,6,Monaco Grand Prix,Max Verstappen,0,18,R,Accident
2786,2016,9,Austrian Grand Prix,Felipe Massa,0,20,20,Brakes
2788,2016,9,Austrian Grand Prix,Daniil Kvyat,0,22,R,Retired
3069,2017,1,Australian Grand Prix,Daniel Ricciardo,0,17,R,Power Unit


In [9]:

# APEX — Conservative race-start audit

import pandas as pd

audit_df = all_results_df.copy()

audit_df["status_normalized"] = (
    audit_df["status"].astype("string").str.strip().str.lower()
)

audit_df["laps_numeric"] = pd.to_numeric(
    audit_df["laps_completed"], errors="coerce"
)

# Only explicitly non-starting / non-qualifying statuses are classified
# as confirmed non-starters by this first-pass rule.
explicit_nonstarter_statuses = {
    "did not start",
    "withdrew",
    "failed to qualify",
    "did not qualify",
}

explicit_nonstarter = audit_df["status_normalized"].isin(
    explicit_nonstarter_statuses
)

evidence_of_start = (
    ~explicit_nonstarter
    & audit_df["laps_numeric"].gt(0)
)

audit_df["start_audit"] = "NEEDS_REVIEW"
audit_df.loc[evidence_of_start, "start_audit"] = "EVIDENCE_OF_START"
audit_df.loc[explicit_nonstarter, "start_audit"] = (
    "EXPLICIT_NONSTART_STATUS"
)

print("Start-audit categories:")
display(
    audit_df["start_audit"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="records")
)

print("\nStatus breakdown for explicit non-starts:")
display(
    audit_df.loc[explicit_nonstarter, "status_normalized"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="records")
)

print("\nCases needing review:")
review_columns = [
    "season", "round", "race_name", "driver_name",
    "grid_position", "laps_completed", "finish_position",
    "position_text", "status",
]

needs_review = audit_df.loc[
    audit_df["start_audit"].eq("NEEDS_REVIEW"),
    review_columns,
]

display(needs_review.head(40))

print("\nGrid-zero summary by audit category:")
display(
    audit_df.loc[audit_df["grid_position"].eq(0)]
    .groupby(["start_audit", "status"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)


Start-audit categories:


,category,records
0,EVIDENCE_OF_START,7062
1,NEEDS_REVIEW,193
2,EXPLICIT_NONSTART_STATUS,30



Status breakdown for explicit non-starts:


,status,records
0,did not start,16
1,withdrew,14



Cases needing review:


,season,round,race_name,driver_name,grid_position,laps_completed,finish_position,position_text,status
44,2010,2,Australian Grand Prix,Sébastien Buemi,12,0,21,R,Collision
45,2010,2,Australian Grand Prix,Nico Hülkenberg,15,0,22,R,Collision
46,2010,2,Australian Grand Prix,Kamui Kobayashi,16,0,23,R,Collision
47,2010,2,Australian Grand Prix,Jarno Trulli,20,0,24,W,Hydraulics
71,2010,3,Malaysian Grand Prix,Pedro de la Rosa,12,0,24,W,Engine
92,2010,4,Chinese Grand Prix,Sébastien Buemi,13,0,21,R,Accident
93,2010,4,Chinese Grand Prix,Kamui Kobayashi,15,0,22,R,Accident
94,2010,4,Chinese Grand Prix,Vitantonio Liuzzi,18,0,23,R,Accident
95,2010,4,Chinese Grand Prix,Timo Glock,19,0,24,W,Engine
118,2010,5,Spanish Grand Prix,Bruno Senna,21,0,23,R,Accident



Grid-zero summary by audit category:


,start_audit,status,records
0,EVIDENCE_OF_START,+1 Lap,18
6,EVIDENCE_OF_START,Finished,11
1,EVIDENCE_OF_START,+2 Laps,6
13,EXPLICIT_NONSTART_STATUS,Withdrew,4
10,EVIDENCE_OF_START,Retired,2
9,EVIDENCE_OF_START,Power Unit,2
5,EVIDENCE_OF_START,Collision damage,2
7,EVIDENCE_OF_START,Gearbox,1
8,EVIDENCE_OF_START,Lapped,1
4,EVIDENCE_OF_START,Collision,1


In [10]:

# APEX — Second-pass audit of zero-lap records

audit_df = all_results_df.copy()

audit_df["status_normalized"] = (
    audit_df["status"].astype("string").str.strip().str.lower()
)

audit_df["position_text_normalized"] = (
    audit_df["position_text"].astype("string").str.strip().str.upper()
)

audit_df["laps_numeric"] = pd.to_numeric(
    audit_df["laps_completed"], errors="coerce"
)

# Recreate the existing conservative review group.
explicit_nonstarter_statuses = {
    "did not start",
    "withdrew",
    "failed to qualify",
    "did not qualify",
}

explicit_nonstarter = (
    audit_df["status_normalized"].isin(explicit_nonstarter_statuses)
    | audit_df["position_text_normalized"].isin({"W", "F"})
)

needs_review = audit_df.loc[
    ~explicit_nonstarter & audit_df["laps_numeric"].eq(0)
].copy()

print("Unresolved zero-lap records:", len(needs_review))

# Summarize official code + reported status.
review_groups = (
    needs_review.groupby(
        ["position_text_normalized", "status"],
        dropna=False,
    )
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)

display(review_groups.head(40))

print("\nUnresolved cases by position code:")
display(
    needs_review["position_text_normalized"]
    .value_counts(dropna=False)
    .rename_axis("position_text")
    .reset_index(name="records")
)

print("\nGrid-zero cases by official position code:")
display(
    audit_df.loc[audit_df["grid_position"].eq(0)]
    .groupby(["position_text_normalized", "status"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)


Unresolved zero-lap records: 171


,position_text_normalized,status,records
4,R,Collision,86
1,R,Accident,32
11,R,Retired,32
0,D,Disqualified,5
5,R,Collision damage,5
8,R,Hydraulics,2
9,R,Power Unit,2
12,R,Suspension,2
2,R,Battery,1
3,R,Brakes,1



Unresolved cases by position code:


,position_text,records
0,R,166
1,D,5



Grid-zero cases by official position code:


,position_text_normalized,status,records
0,10,+1 Lap,4
35,W,Withdrew,4
5,13,+1 Lap,3
11,16,+1 Lap,3
29,R,Retired,2
4,12,+1 Lap,2
6,13,Finished,2
22,5,Finished,2
9,15,+1 Lap,2
12,16,+2 Laps,2
